In [7]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#loading data
import pandas as pd
train = pd.read_csv("final_data/train/dico_nli_train_track1_participant_labeled.csv")
dev = pd.read_csv("final_data/dev/dico_nli_dev_track1_participant_labeled.csv")
ref = pd.read_csv("final_data/dev/dico_nli_dev_track1_reference.csv")

print(train.columns.tolist())
print(ref.columns.tolist())
train.head()

['instance_id', 'pair_id', 'text1_lang', 'text2_lang', 'text1', 'text2', 'label']
['instance_id', 'pair_id', 'text1_lang', 'text2_lang', 'text1', 'text2', 'reverse_pair_id', 'label']


,instance_id,pair_id,text1_lang,text2_lang,text1,text2,label
0,dico_phrasis_0000001__en-en__flipped,dico_phrasis_0000001,en,en,Conservative opposition,Conservatives,FORWARD_ENTAILMENT
1,dico_phrasis_0000001__en-en__original,dico_phrasis_0000001,en,en,Conservatives,Conservative opposition,BACKWARD_ENTAILMENT
2,dico_phrasis_0000003__en-en__flipped,dico_phrasis_0000003,en,en,Japanese PM Shinzo Abe,Abe,FORWARD_ENTAILMENT
3,dico_phrasis_0000003__en-en__original,dico_phrasis_0000003,en,en,Abe,Japanese PM Shinzo Abe,BACKWARD_ENTAILMENT
4,dico_phrasis_0000005__en-en__flipped,dico_phrasis_0000005,en,en,ballistic missiles,missiles,FORWARD_ENTAILMENT


In [ ]:
#The label distribution in train and dev.
print(train['label'].value_counts())
print(dev['label'].value_counts())

label
FORWARD_ENTAILMENT     881
BACKWARD_ENTAILMENT    881
EQUIVALENCE            802
NEGATIVE_OTHER         478
Name: count, dtype: int64
label
FORWARD_ENTAILMENT     190
BACKWARD_ENTAILMENT    190
EQUIVALENCE            174
NEGATIVE_OTHER         106
Name: count, dtype: int64


In [ ]:
#Three examples of each label, chosen by you, with one sentence on why each has its label
label = "FORWARD_ENTAILMENT"
train[train["label"] == label][["text1", "text2"]].sample(10, random_state=0)

,text1,text2
300,A young boy,A boy
812,"jeans , a dark bikini top , and a bracelet",with a bracelet a bikini top and jeans
1829,Malaysian jet,jet
202,through a snowy field,through snow
1971,A young boy,A boy
2205,in Pakistan,Pakistan
2433,A home office,Room
2469,on blue sofa,on the sofa
790,A black and white dog,A dog
62,of February general election,general election


EXAMPLE 1: "A young boy" and "A boy" - This is forward entailment because text1 adds the modifier young making it specific. Every young boy is a boy but not every boy is young.

EXAMPLE 2: "A black and white dog" and "A dog" - This is forward entailment because every black and white dog is a dog but not every dog is black and white.

EXAMPLE 3: "Malayasian jet" and "jet" - This is forward entailment because every Malayasian jet is a jet but not every jet is Malaysian.


In [14]:
#Three examples of each label, chosen by you, with one sentence on why each has its label
label = "BACKWARD_ENTAILMENT"
train[train["label"] == label][["text1", "text2"]].sample(10, random_state=0)

,text1,text2
301,A boy,A young boy
813,with a bracelet a bikini top and jeans,"jeans , a dark bikini top , and a bracelet"
1828,jet,Malaysian jet
203,through snow,through a snowy field
1970,A boy,A young boy
2204,Pakistan,in Pakistan
2432,Room,A home office
2468,on the sofa,on blue sofa
791,A dog,A black and white dog
63,general election,of February general election


EXAMPLE 1: "on the sofa" and "on blue sofa" - This is backward entailment because text2 adds the modifier "blue", making it more specific. Every blue sofa is a sofa, but not every sofa is blue.

EXAMPLE 2: "general election" and "of February general election" - This is backward entailment because text2 adds the detail "February". Every February general election is a general election, but not every general election is in February.

EXAMPLE 3: "Pakistan" and "in Pakistan" - This is backward entailment because text2 adds the preposition "in", which narrows the phrase to a location within Pakistan.

In [15]:
#Three examples of each label, chosen by you, with one sentence on why each has its label
label = "EQUIVALENCE"
train[train["label"] == label][["text1", "text2"]].sample(10, random_state=0)

,text1,text2
898,Ukraine protesters,Ukrainian protesters
1143,to another man,toward a second man
975,concerning negotiations over nuclear matters,for nuclear talks
1312,to Take Over,assumes
904,S. Korean president,South Korean President
1330,among dead,killed
1543,The man,the male adult
1411,into Lebanon,entering Lebanese territory
1609,maintaining an upright stance,is standing
1413,An Apple computer,A Macintosh computer


EXAMPLE 1: "S. Korean president" and "South Korean President" - This is equivalence because "S." is just an abbreviation of "South", so both phrases refer to exactly the same person and neither adds any information.

EXAMPLE 2: "maintaining an upright stance" and "is standing" - This is equivalence because text1 is a longer paraphrase of text2: keeping an upright stance is what standing means.

EXAMPLE 3: "The man" and "the male adult" - This is equivalence because "man" is defined as an adult male, so the two phrases describe the same thing in different words.

In [16]:
#Three examples of each label, chosen by you, with one sentence on why each has its label
label = "NEGATIVE_OTHER"
train[train["label"] == label][["text1", "text2"]].sample(10, random_state=0)

,text1,text2
2579,Russian Prime Minister,Russia
2688,by Gibraltar/Spain row,over Gibraltar events
2752,A young child,An Asian child
2719,' bunga bunga ' witnesses,witnesses in prostitution trial
2942,of a person,on a person
2705,Suicide bomber,Suicide bomb
2706,to 6.2035,to 6.1408
2925,on side of street,on a crowded street
2677,10 Brotherhood supporters,supporters
2585,in Syria,in Israeli raid


EXAMPLE 1: "A young child" and "An Asian child" - This is negative because both phrases describe a child but add different, unrelated details. A young child is not necessarily Asian and an Asian child is not necessarily young

EXAMPLE 2: "to 6.2035" and "to 6.1408" - This is negative because the two phrases have the same structure but different numbers. They are the same kind of thing, but one value does not imply the other.

EXAMPLE 3: "Russian Prime Minister" and "Russia" - This is negative because the phrases are related but refer to different things: one is a person and the other is a country. Neither being true tells you the other is.

In [20]:
#How reversed pairs are linked in the data (look at the instance IDs and the reverse_pair_id column in the reference file).
print(ref.columns.tolist)
ref[ref['instance_id'].str.contains("0000002")]


<bound method IndexOpsMixin.tolist of Index(['instance_id', 'pair_id', 'text1_lang', 'text2_lang', 'text1', 'text2',
       'reverse_pair_id', 'label'],
      dtype='str')>


,instance_id,pair_id,text1_lang,text2_lang,text1,text2,reverse_pair_id,label
0,dico_phrasis_0000002__en-en__flipped,dico_phrasis_0000002,en,en,'s war shrine visit,'s shrine visit,dico_phrasis_0000002__en-en__original,FORWARD_ENTAILMENT
1,dico_phrasis_0000002__en-en__original,dico_phrasis_0000002,en,en,'s shrine visit,'s war shrine visit,dico_phrasis_0000002__en-en__flipped,BACKWARD_ENTAILMENT


In [19]:
ref.groupby("label")["reverse_pair_id"].apply(lambda s: s.notna().sum())

label
BACKWARD_ENTAILMENT    190
EQUIVALENCE            174
FORWARD_ENTAILMENT     190
NEGATIVE_OTHER           0
Name: reverse_pair_id, dtype: int64

Every pair of phrases is in the data twice: once in the normal order and once with the two phrases swapped. 
Both rows have the same pair_id, which is how you know they belong together. The instance_id is the pair_id plus an ending that says which version it is: __original or __flipped. The reverse_pair_id column gives the instance_id of the other row in the pair. When the phrases are swapped, the label changes to match. FORWARD_ENTAILMENT becomes BACKWARD_ENTAILMENT, and BACKWARD becomes FORWARD. EQUIVALENCE stays the same. NEGATIVE_OTHER rows do not have a swapped partner. That is why the scorer checks consistency on only 277 pairs instead of all the data.